![](https://mcd.unison.mx/wp-content/themes/awaken/img/logo_mcd.png)

# A la búsqueda del mejor modelo de clasificación

## Aprendizaje Automático Aplicado

### Maestría en Ciencia de Datos

#### **Julio Waissman**, 2026

[**Abrir en google Colab**](https://colab.research.google.com/github/mcd-unison/aaa-curso/blob/main/ejemplos/ejemplo_optuna.ipynb)


## Configuración y conección a DataBricks

Vamos a hacerlo en Databrick y lo estamos poniendo como si lo ejecutaramos en Colab. Si no es el caso, es necesario obtener la información específica de conexión a DataBricks y configurar el entorno de ejecución para poder registrar los experimentos, modelos y métricas, tal como lo vimos en [esta libreta](https://github.com/mcd-unison/aaa-curso/blob/main/ejemplos/mlflow3-et.ipynb).

In [ ]:
!pip install --upgrade "mlflow>=3.1"

In [ ]:
from google.colab import userdata
import os

DATABRICKS_TOKEN = userdata.get('DATABRICKS_TOKEN')
DATABRICKS_HOST = "https://dbc-193e0e35-8e4c.cloud.databricks.com" 
MLFLOW_TRACKING_URI = "databricks"
MLFLOW_EXPERIMENT_ID = "403957687974907"

os.environ['DATABRICKS_TOKEN'] = DATABRICKS_TOKEN
os.environ['DATABRICKS_HOST'] = DATABRICKS_HOST
os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI
os.environ['MLFLOW_EXPERIMENT_ID'] = MLFLOW_EXPERIMENT_ID

import mlflow

## Obteniendo el conjunto de datos

Para este ejemplo vamos a utilizar el conjunto de datos [Adult](https://www.openml.org/search?type=data&status=active&id=1590&sort=runs) de [OpenML](https://www.openml.org/), que es un conjunto de datos clásico para problemas de clasificación. El objetivo es predecir si una persona gana más de 50K al año basándose en características como la edad, el nivel educativo, el estado civil, entre otras.

In [ ]:
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

# Cargar datos
data = fetch_openml(data_id=1590, as_frame=True, parser='auto')
df = data.frame

# Definir objetivo y características
target = 'class'
X = df.drop(columns=[target])
y = df[target].apply(lambda x: 1 if x == '>50K' else 0)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

y podemos ver un poco que tenemos:

1. Atributos categóricos
2. Atributos numñericos
3. Un problema de clasificación binaria con clases desbalanceadas,, pero no mucho.
4. Variables con valores faltantes, lo que nos da la oportunidad de practicar técnicas de imputación.

In [ ]:
print("Información básica de X:")
print(X.info())
print("-"*40 + "\n Información básica de y:")
print(y.value_counts())

## Preprocesamiento de datos

Vamos a realizar un `ColumnTransformer` para preprocesar los datos, donde vamos a imputar los valores faltantes utilizando la mediana para las variables numéricas y la moda para las variables categóricas. Además, vamos a escalar las variables numéricas utilizando `StandardScaler` y vamos a codificar las variables categóricas utilizando `OneHotEncoder`.

Todo esto lo vamos a agregar en un `pipeline` junto al modelo, de manera que el modelo de aprendizaje encapsule cuando se utilice tanto el preprocesamiento cono el modelo de clasificación.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def get_preprocessor(X):
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns
    cat_cols = X.select_dtypes(include=['category', 'object']).columns

    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])

    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ])

    return ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

## Función Objetivo de Optuna con Nested Runs

Para que cada prueba de Optuna sea visible individualmente en Databricks, usamos nested=True dentro del bucle de optimización.

Aqui vamos a especificar como vamos a buscar el mejor modelo de clasificación, definiendo una función objetivo que:
1. Sugerirá un modelo de clasificación y sus hiperparámetros utilizando `trial.suggest_categorical` y `trial.suggest_float` o `trial.suggest_int`.
2. Creará un pipeline que incluya el preprocesamiento y el modelo.
3. Iniciará un nested run en MLflow para cada prueba, lo que permitirá registrar los parámetros, métricas y modelos de cada prueba de manera independiente.
4. Entrenará el modelo y evaluará su desempeño utilizando el F1-score, que es una métrica adecuada para problemas de clasificación con clases desbalanceadas.
5. Registrará el F1-score en MLflow para cada prueba.

In [ ]:
!pip install optuna

In [ ]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

def objective(trial):
    # 1. Definir hiperparámetros por modelo y establecer el modelo de aprendizaje
    classifier_name = trial.suggest_categorical(
        "classifier", ["LogReg", "SVC", "RandomForest", "XGBoost"]
    )
    
    if classifier_name == "LogReg":
        model = LogisticRegression(
            C=trial.suggest_float("C", 0.1, 10.0, log=True)
        )
    elif classifier_name == "SVC":
        model = SVC(
            C=trial.suggest_float("svc_C", 0.1, 10.0, log=True), 
            kernel='rbf'
        )
    elif classifier_name == "RandomForest":
        model = RandomForestClassifier(
            max_depth=trial.suggest_int("max_depth", 4, 15),
            n_estimators=trial.suggest_int("n_estimators", 50, 200)
        )
    else:
        model = XGBClassifier(
            learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2),
            n_estimators=trial.suggest_int("xgb_estimators", 50, 150)
        )

    # 2. Crear Pipeline completo
    preprocessor = get_preprocessor(X_train)
    clf = Pipeline(steps=[
        ('preprocessor', preprocessor), 
        ('classifier', model)
        ]
    )

    # 3. Iniciar un Nested Run en MLflow para este Trial específico
    # El nombre del run incluirá el número de intento para facilitar la lectura
    
    run_name=f"Trial_{trial.number}_{classifier_name}"
    with mlflow.start_run(run_name=run_name, nested=True):
        clf.fit(X_train, y_train)
        
        # Evaluación
        preds = clf.predict(X_test)
        score = f1_score(y_test, preds)
        
        # MLflow Autolog ya guardó casi todo, pero podemos añadir etiquetas extra
        mlflow.set_tag("model_type", classifier_name)
        mlflow.log_metric("f1_test", score)
        
        return score

## Ejecuta el aprendizaje y búsqueda de mejor modelo

Para ejecutar la optimización, simplemente llamamos a `study.optimize` dentro de un run maestro (parent run) en MLflow. Esto nos permitirá tener un registro centralizado de toda la optimización, con cada prueba individual registrada como un nested run. 

In [ ]:
# ACTIVAR AUTOLOG: Esto registrará modelos, parámetros y métricas automáticamente
mlflow.sklearn.autolog(log_models=True)
mlflow.xgboost.autolog(log_models=True)

# Ejecutar la optimización dentro de un Run Maestro (Parent Run)
with mlflow.start_run(run_name="Optuna_Master_Optimization"):
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=15) # Ajusta n_trials según tu tiempo

    # Registrar el mejor resultado en el run maestro
    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_f1_score", study.best_value)
    print(f"Optimización completada. Mejor F1: {study.best_value}")

Al finalizar la optimización, también registraremos el mejor resultado encontrado en el run maestro para facilitar su consulta y comparación con otras optimizaciones que podamos realizar en el futuro.

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Entrenar el modelo ganador final basado en los mejores parámetros de Optuna
best_params = study.best_params
classifier_name = best_params.pop('classifier')

if classifier_name == "LogReg":
    final_model = LogisticRegression(**best_params)
elif classifier_name == "RandomForest":
    final_model = RandomForestClassifier(**best_params)
else:
    final_model = XGBClassifier(**best_params)

final_pipeline = Pipeline(steps=[('preprocessor', get_preprocessor(X_train)), ('classifier', final_model)])

with mlflow.start_run(run_name="MODELO_FINAL_PRODUCCION"):
    final_pipeline.fit(X_train, y_train)
    y_pred = final_pipeline.predict(X_test)
    
    # Reporte de métricas finales
    print(classification_report(y_test, y_pred))
    
    # Matriz de Confusión Autoguardada por Autolog
    ConfusionMatrixDisplay.from_estimator(final_pipeline, X_test, y_test)
    plt.title("Matriz de Confusión - Mejor Modelo")
    plt.show()

## Para completar el ejercicio

1. Mejora la funcion objetivo, al incluir mñas hiperparámetros, o modificando el rango donde se busca a cada uno.
2. Convierte desde la lectura a todas las variables numñericas a flotantes, para mejor el esquema de modelo para su futura puesta en producción.
3. Agrega una validación cruzada dentro de la función objetivo, para obtener una evaluación más robusta del desempeño de cada modelo.
4. Agrega el modelo de CatBoost a la búsqueda de modelos, y ajusta los hiperparámetros que se le pasan a este modelo.


## Lo que se debería haber hecho

1. Análisis exploratorio de datos para entender mejor el conjunto de datos y sus características.
2. Modelo por regresión logística, probar hacer ingeniería de caractarísticas y ajustarlo a mano para un primer baseline decente.
3. Ahora si, en base a lo que se sabe, seleccionar los modelos a utilizar y los hiperparámetros a ajustar, y luego ejecutar la optimización con Optuna.
4. Analizar los resultados obtenidos, tanto a nivel de métricas como a nivel de modelos, para entender qué modelos y qué hiperparámetros están funcionando mejor, y por qué.
5. Rerajustar la función objetivo y repetir el proceso de optimización para intentar mejorar los resultados obtenidos.
6. Finalmente, seleccionar el mejor modelo encontrado y prepararlo para su puesta en producción.